# Lesson 04 - Tool Use デザインパターン

このレッスンでは、Microsoft Agent Framework（Python）を使ったAIエージェントの**Tool Use**デザインパターンを学びます。内容は以下の通りです:

- `@tool`デコレーターと型付きパラメータを使った関数ツールの定義
- モデルが各ツールの機能を理解できるようにツールスキーマを提供すること
- `approval_mode`によるツール実行の制御
- Pydanticモデルと`response_format`を使った**構造化された出力**の返却

シナリオは、目的地の検索、空き状況の確認、フライト情報の取得ができる**旅行予約エージェント**です。


## セットアップ


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @toolデコレーターでツールを定義する

`@tool`デコレーターは、単純なPython関数をエージェントが呼び出せるツールに変えます。  
重要なポイント：

- **ドキュメンテーション文字列**がモデルが見るツールの説明になります。  
- **型注釈**（説明付きの`Annotated`を含む）がツールのスキーマを定義します。  
- `approval_mode`は呼び出す前にユーザーの承認が必要かどうかを制御します。


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## 複数のツールを持つエージェントの作成

ユーザーの質問に答えるためにモデルが必要なツールを呼び出せるように、すべての3つのツールをクライアントに渡します。


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## ツールを使用した構造化出力

`response_format` をPydanticモデルに設定することで、エージェントは自由形式のテキストではなく、型が明確なJSONオブジェクトを返すよう強制されます。これは下流のコードが結果をプログラム的に利用する必要がある場合に便利です。


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## ツール承認パターン

`@tool` の `approval_mode` パラメーターは、ツールの呼び出しが実行前に人間の承認を必要とするかどうかを制御します:

| モード | 挙動 |
|---|---|
| `"never_require"` | ツールは自動的に実行されます — ユーザーの確認は不要です。 |
| `"always_require"` | すべての呼び出しは実行前にユーザーの承認が必要です。 |

副作用のあるツール（例: 飛行機の予約、クレジットカードの課金）には `"always_require"` を使用して、人間が介入できるようにしてください。


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## 概要

このレッスンでは、以下のことを学びました：

1. 型付きパラメーターとツールスキーマとして機能するドキュメンテーション文字列を用いて、`@tool`デコレーターで**ツールを定義する**方法。
2. エージェントが複雑なクエリに回答するために複数のツールを順番に呼び出せるように**ツールを組み合わせる**方法。
3. Pydanticモデルを`response_format`として渡すことで、**構造化された出力を返す**方法。
4. `approval_mode`でツール承認を制御し、重要な操作で人間の関与を維持する方法。

これらのパターンは、外部システムと安全に連携できる信頼性の高い本番環境対応エージェントを構築するための基礎となります。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類はAI翻訳サービス[Co-op Translator](https://github.com/Azure/co-op-translator)を使用して翻訳されました。正確性を期しておりますが、自動翻訳には誤りや不正確な表現が含まれる可能性があります。原文は原言語版を権威ある情報源としてご参照ください。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や誤訳についても責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
